# 14 — Format Disaggregation Robustness Check

This notebook disaggregates the 282-store analytical sample by
**store format (6-digit NAICS)** and re-runs the full Huff/PSO calibration separately for the two dominant
formats, so the parameter trends can be compared against the pooled result.

**Tasks (three stages):**
1. **Store→format mapping.** Resolve each of the 282 stores to a NAICS code / category by joining
   `table_2018.csv`'s `B_store` (SafeGraph `safegraph_place_id`) to `nyc-poi-info.csv`. Priority:
   exact `safegraph_place_id` (`place_id`) → `safegraph_brand_id` (`brand_id`) → brand name
   (`brand_name`) → `unresolved`. Two formats dominate: **452210 Department Stores** and
   **452319 General Merchandise Stores**.
2. **Stratified sample build.** For each format, find CBGs with nonzero visits in all four years,
   then draw a ~450-CBG sample stratified by borough × income quintile (seed 42), reusing the exact
   global min–max normalisation (→ [1,28]) of the main analysis.
3. **By-format PSO calibration.** Run the identical PSO (20 particles, 6 dims, c1=c2=1.5, w=0.9,
   exponent bounds [1,15], 10 iters) per CBG per year, and compute the mean-parameter % changes and
   fit, comparing to the pooled result.

**Reproducibility / safety.** All stochastic steps are seeded (sampling seed 42; PSO `np.random.seed(42)`).
`WRITE_OUTPUTS=False` so running this notebook **recomputes everything in memory and verifies against the
already-saved artefacts in `outputs/format_disaggregation/` without modifying them.**


In [1]:
import os, numpy as np, pandas as pd, logging
logging.getLogger('pyswarms').setLevel(logging.ERROR)
from pyswarms.single.global_best import GlobalBestPSO
pd.set_option('display.width',160); pd.set_option('display.max_columns',None)

REPO='/Users/mohsenbahrami/Desktop/frontiers_repo'
DATA=os.path.join(REPO,'data/model_inputs')
POI =os.path.join(REPO,'data/processed/nyc-poi-info.csv')
OUT =os.path.join(REPO,'outputs/format_disaggregation')   # renamed dir (formerly item3_*)
YEARS=[2018,2019,2020,2021]
NORM=28          # feature-normalisation ceiling  -> features scaled to [1,28]
PSO_RANGE=15     # PSO exponent search bound       -> exponents in [1,15]
SEED=42
WRITE_OUTPUTS=False   # keep False: verify-only, never overwrite saved outputs

PARAM=['H_Area_of_store','R_Percentage_of_Visits_by_brand','J_POI_count_where_store_is',
       'K_POI_diversity_where_store_is','L_Demographic_similarity','G_Distance_between_cbg_and_store']
PRETTY={'H_Area_of_store':'Store area','R_Percentage_of_Visits_by_brand':'Chain loyalty',
        'J_POI_count_where_store_is':'POI count','K_POI_diversity_where_store_is':'POI diversity',
        'L_Demographic_similarity':'Demographic similarity','G_Distance_between_cbg_and_store':'Distance'}
BOROUGH={5:'Bronx',47:'Brooklyn',61:'Manhattan',81:'Queens',85:'Staten Island'}
def borough_of(c): return BOROUGH.get((c//10**7)%1000,'Unknown')
def convert(x,lo,hi): return ((x-lo)/(hi-lo))*(NORM-1)+1 if hi!=lo else pd.Series(1.0,index=x.index)
PASS=[]   # collects verification results
def check(name,ok,detail=''):
    PASS.append((name,ok,detail)); print(('PASS ' if ok else '*** FAIL *** ')+name+('  '+detail if detail else ''))

## Stage 1a — Store → format (NAICS) mapping

In [2]:
# --- build consolidated store->NAICS mapping (place_id > brand_id > brand_name > unresolved) ---
poi=pd.read_csv(POI, low_memory=False)
t18=pd.read_csv(os.path.join(DATA,'table_2018.csv'),usecols=['B_store','O_Brand_name','P_safegraph_brand_id'])
st=t18.drop_duplicates('B_store')[['B_store','O_Brand_name','P_safegraph_brand_id']].reset_index(drop=True)

# place-level lookup keyed on safegraph_place_id
pk=poi.drop_duplicates('safegraph_place_id').set_index('safegraph_place_id')
spid=set(poi['safegraph_place_id'].astype(str))

# brand-id lookup: explode comma-joined safegraph_brand_ids, take single consistent NAICS
pe=poi.dropna(subset=['safegraph_brand_ids']).copy()
pe['bid']=pe['safegraph_brand_ids'].str.split(','); pe=pe.explode('bid'); pe['bid']=pe['bid'].str.strip()
def brand_agg(g):
    n=sorted(g['naics_code'].dropna().unique())
    return pd.Series({'naics_code':(int(n[0]) if len(n)==1 else None),
        'top_category':g['top_category'].mode().iloc[0] if g['top_category'].notna().any() else None,
        'sub_category':g['sub_category'].mode().iloc[0] if g['sub_category'].notna().any() else None})
by_bid=pe.groupby('bid').apply(brand_agg,include_groups=False)
pn=poi.dropna(subset=['brands']).copy()
by_name=pn.groupby('brands').apply(brand_agg,include_groups=False)

def resolve(row):
    sid=str(row['B_store'])
    if sid in spid:
        r=pk.loc[sid]
        if pd.notna(r['naics_code']):
            return int(r['naics_code']),r['top_category'],r['sub_category'],'place_id'
    bid=row['P_safegraph_brand_id']
    if bid in by_bid.index and pd.notna(by_bid.loc[bid,'naics_code']):
        a=by_bid.loc[bid]; return int(a['naics_code']),a['top_category'],a['sub_category'],'brand_id'
    bn=row['O_Brand_name']
    if bn in by_name.index and pd.notna(by_name.loc[bn,'naics_code']):
        a=by_name.loc[bn]; return int(a['naics_code']),a['top_category'],a['sub_category'],'brand_name'
    return None,None,None,'unresolved'

res=st.apply(resolve,axis=1,result_type='expand'); res.columns=['naics_code','top_category','sub_category','match_type']
mapping=pd.concat([st,res],axis=1).rename(columns={'B_store':'store_id','O_Brand_name':'brand_name'})
mapping=mapping[['store_id','naics_code','top_category','sub_category','brand_name','match_type']]

print('match_type counts:', mapping['match_type'].value_counts().to_dict())
print('NAICS counts     :', mapping['naics_code'].value_counts(dropna=False).to_dict())

match_type counts: {'brand_id': 195, 'place_id': 47, 'unresolved': 40}
NAICS counts     : {452319.0: 154, 452210.0: 88, nan: 40}


In [3]:
# --- VERIFY mapping against saved store_category_mapping.csv ---
saved_map=pd.read_csv(os.path.join(OUT,'store_category_mapping.csv'))
n210=int((mapping['naics_code']==452210).sum()); n319=int((mapping['naics_code']==452319).sum())
nun =int((mapping['match_type']=='unresolved').sum())
check('mapping counts 88/154/40', (n210,n319,nun)==(88,154,40), f'got {n210}/{n319}/{nun}')
# per-store match_type + naics identical to saved (align on store_id)
m=mapping.set_index('store_id').sort_index(); s=saved_map.set_index('store_id').sort_index()
same_mt=(m['match_type']==s['match_type']).all()
same_na=m['naics_code'].fillna(-1).astype(int).equals(s['naics_code'].fillna(-1).astype(int))
check('per-store match_type identical to saved', bool(same_mt))
check('per-store naics identical to saved', bool(same_na))
dept=set(mapping.loc[mapping['naics_code']==452210,'store_id'])
gen =set(mapping.loc[mapping['naics_code']==452319,'store_id'])

PASS mapping counts 88/154/40  got 88/154/40
PASS per-store match_type identical to saved
PASS per-store naics identical to saved


## Stage 1b — Global normalisation bounds, eligible pools, stratified samples

In [4]:
# global min/max per year over the FULL table (all 282 stores); per-format per-CBG visit totals
bounds={}; visit_tot={452210:{},452319:{}}; income=None
for y in YEARS:
    uc=['A_cbg','B_store',f'D_Number_of_Visits_{y}','M_Median_Income_in_this_cbg',
        'H_Area_of_store',f'R_Percentage_of_Visits_by_brand_{y}','J_POI_count_where_store_is',
        'K_POI_diversity_where_store_is','L_Demographic_similarity','G_Distance_between_cbg_and_store']
    t=pd.read_csv(os.path.join(DATA,f'table_{y}.csv'),usecols=list(dict.fromkeys(uc))).rename(
        columns={f'R_Percentage_of_Visits_by_brand_{y}':'R_Percentage_of_Visits_by_brand',
                 f'D_Number_of_Visits_{y}':'D'})
    bounds[y]={a:(float(t[a].min()),float(t[a].max())) for a in PARAM}
    for fmt,ss in [(452210,dept),(452319,gen)]:
        visit_tot[fmt][y]=t[t['B_store'].isin(ss)].groupby('A_cbg')['D'].sum()
    if y==2018:
        income=t.groupby('A_cbg')['M_Median_Income_in_this_cbg'].first()
    del t
boroughs=pd.Series({c:borough_of(c) for c in income.index})
print('normalisation bounds computed for', list(bounds))

normalisation bounds computed for [2018, 2019, 2020, 2021]


In [5]:
# eligible pool = CBGs with >0 format visits in ALL four years; then stratified ~450 sample (seed 42)
def stratified(pool_cbgs):
    df=pd.DataFrame({'A_cbg':pool_cbgs})
    df['borough']=df['A_cbg'].map(boroughs); df['income']=df['A_cbg'].map(income)
    df['quintile']=pd.qcut(df['income'],5,labels=['Q1','Q2','Q3','Q4','Q5']).astype(str)
    n_pool=len(df); picks=[]
    for (b,q),g in df.groupby(['borough','quintile']):
        n_s=min(int(round(450*len(g)/n_pool)),len(g))
        if n_s>0: picks.append(g.sample(n=n_s,random_state=SEED))
    return df, pd.concat(picks).reset_index(drop=True)

eligible={}; pools={}; samples={}
allc=list(income.index)
for fmt in (452210,452319):
    ok=None
    for y in YEARS:
        vt=visit_tot[fmt][y].reindex(allc).fillna(0); pos=set(vt[vt>0].index)
        ok=pos if ok is None else ok&pos
    eligible[fmt]=sorted(ok)
    pools[fmt],samples[fmt]=stratified(eligible[fmt])
    print(f'format {fmt}: eligible={len(eligible[fmt])}  sampled={len(samples[fmt])}')

format 452210: eligible=4997  sampled=448
format 452319: eligible=5276  sampled=450


In [6]:
# --- VERIFY pools/samples vs saved sample files ---
check('eligible pools 4997 / 5276', (len(eligible[452210]),len(eligible[452319]))==(4997,5276),
      f'got {len(eligible[452210])}/{len(eligible[452319])}')
check('sample sizes 448 / 450', (len(samples[452210]),len(samples[452319]))==(448,450),
      f'got {len(samples[452210])}/{len(samples[452319])}')
for fmt,f in [(452210,'sample_452210_department_stores.csv'),(452319,'sample_452319_general_merchandise.csv')]:
    saved_cbgs=set(pd.read_csv(os.path.join(OUT,f))['A_cbg'].unique())
    check(f'{fmt} sampled CBG set matches saved', set(samples[fmt]['A_cbg'])==saved_cbgs)

PASS eligible pools 4997 / 5276  got 4997/5276
PASS sample sizes 448 / 450  got 448/450
PASS 452210 sampled CBG set matches saved


PASS 452319 sampled CBG set matches saved


## Stage 2 — By-format PSO calibration

Uses the **saved** sample files as PSO inputs (identical to the original run), so parameters reproduce
exactly. PSO order and single global `np.random.seed(42)` replicate the original call sequence.

In [7]:
OPTIONS={'c1':1.5,'c2':1.5,'w':0.9}; BOUNDS=(np.array([1.]*6),np.array([float(PSO_RANGE)]*6))
def run_cbg(g):
    H,R,J,K,L,G=[g[c].values.astype(float) for c in PARAM]
    actual=g['C_Percentage_of_Visits'].values.astype(float)
    if actual.max()==0: return None,None
    am=actual.mean(); asd=actual.std()
    def opt(X):
        out=np.empty(X.shape[0])
        for i in range(X.shape[0]):
            p=X[i]; a=(H**p[0])*(R**p[1])*(J**p[2])*(K**p[3])*(L**p[4])/(G**p[5])
            s=a.sum()
            if s!=0: a=a/s
            bsd=a.std()
            out[i]=1.0 if (asd==0 or bsd==0) else 1.0-np.mean((actual-am)*(a-a.mean()))/(asd*bsd)
        return out
    o=GlobalBestPSO(n_particles=20,dimensions=6,options=OPTIONS,bounds=BOUNDS)
    return o.optimize(opt,iters=10,verbose=False)

SAMP={452210:'sample_452210_department_stores.csv',452319:'sample_452319_general_merchandise.csv'}
data={f:pd.read_csv(os.path.join(OUT,SAMP[f])) for f in SAMP}

np.random.seed(SEED)                       # exactly as the original stage-2 script
results={}                                 # results[fmt][year] -> DataFrame(cbg,cost,6 params)
for fmt in (452210,452319):                # ORDER MATTERS for RNG reproduction: 452210 then 452319
    results[fmt]={}
    for y in YEARS:
        sub=data[fmt][data[fmt]['year']==y]
        rows=[]
        for c in sorted(sub['A_cbg'].unique()):
            cost,v=run_cbg(sub[sub['A_cbg']==c])
            rows.append([c,cost]+list(v))
        results[fmt][y]=pd.DataFrame(rows,columns=['cbg','cost']+PARAM)
    print(f'format {fmt}: PSO done for all years')

format 452210: PSO done for all years


format 452319: PSO done for all years


In [8]:
# --- VERIFY recomputed PSO reproduces the saved per-CBG CSVs (bit-level, allclose) ---
import numpy as np
for fmt in (452210,452319):
    okall=True
    for y in YEARS:
        saved=pd.read_csv(os.path.join(OUT,f'PSO_{fmt}_{y}.csv')).set_index('cbg').sort_index()
        fresh=results[fmt][y].set_index('cbg').sort_index()
        okall &= np.allclose(fresh[['cost']+PARAM].values, saved[['cost']+PARAM].values, rtol=1e-9, atol=1e-9)
    check(f'{fmt} PSO params reproduce saved CSVs (allclose)', okall)

PASS 452210 PSO params reproduce saved CSVs (allclose)
PASS 452319 PSO params reproduce saved CSVs (allclose)


## Stage 2 — Fit and % change, compared to pooled (verification against `stage2_by_format_results.txt`)

In [9]:
# mean fit (1-cost) and cross-CBG parameter means per format-year
def pct(a,b): return (b-a)/a*100
fit={}; means={}
for fmt in (452210,452319):
    fit[fmt]={}; means[fmt]={}
    for y in YEARS:
        r=results[fmt][y]; fit[fmt][y]=1-r['cost'].mean(); means[fmt][y]=r[PARAM].mean()

print('Mean fit (1-cost) per format-year:')
for fmt in (452210,452319):
    print('  ',fmt,{y:round(fit[fmt][y],4) for y in YEARS})

# Expected values copied from saved stage2_by_format_results.txt
EXP_FIT={452210:{2018:0.7132,2019:0.7203,2020:0.2417,2021:0.2104},
         452319:{2018:0.7035,2019:0.7087,2020:0.2381,2021:0.1711}}
for fmt in (452210,452319):
    ok=all(round(fit[fmt][y],4)==EXP_FIT[fmt][y] for y in YEARS)
    check(f'{fmt} fit matches saved report', ok, str({y:round(fit[fmt][y],4) for y in YEARS}))

# %change chain-loyalty & store area (the headline numbers in the report)
EXP_PCT={  # (fmt, y1,y2): {param: expected%}
 (452210,2019,2020):{'Chain loyalty':33.85,'Store area':58.62,'POI count':31.77,'POI diversity':8.20,'Demographic similarity':-1.51,'Distance':-29.06},
 (452210,2019,2021):{'Chain loyalty':35.70,'Store area':54.15,'POI count':61.87,'POI diversity':9.51,'Demographic similarity':-5.82,'Distance':-35.80},
 (452319,2019,2020):{'Chain loyalty':3.41,'Store area':47.47,'POI count':50.12,'POI diversity':25.63,'Demographic similarity':10.25,'Distance':-28.02},
 (452319,2019,2021):{'Chain loyalty':4.91,'Store area':44.45,'POI count':96.24,'POI diversity':10.11,'Demographic similarity':6.81,'Distance':-39.58},
}
print('\n% change (recomputed) vs saved report:')
for (fmt,y1,y2),exp in EXP_PCT.items():
    okpair=True
    for c in PARAM:
        got=round(pct(means[fmt][y1][c],means[fmt][y2][c]),2); want=exp[PRETTY[c]]
        if abs(got-want)>0.01: okpair=False; print(f'   MISMATCH {fmt} {y1}->{y2} {PRETTY[c]}: got {got} want {want}')
    check(f'{fmt} {y1}->{y2} %changes match report', okpair)

Mean fit (1-cost) per format-year:
   452210 {2018: np.float64(0.7132), 2019: np.float64(0.7203), 2020: np.float64(0.2417), 2021: np.float64(0.2104)}
   452319 {2018: np.float64(0.7035), 2019: np.float64(0.7087), 2020: np.float64(0.2381), 2021: np.float64(0.1711)}
PASS 452210 fit matches saved report  {2018: np.float64(0.7132), 2019: np.float64(0.7203), 2020: np.float64(0.2417), 2021: np.float64(0.2104)}
PASS 452319 fit matches saved report  {2018: np.float64(0.7035), 2019: np.float64(0.7087), 2020: np.float64(0.2381), 2021: np.float64(0.1711)}

% change (recomputed) vs saved report:
PASS 452210 2019->2020 %changes match report
PASS 452210 2019->2021 %changes match report
PASS 452319 2019->2020 %changes match report
PASS 452319 2019->2021 %changes match report


## Result

The by-format calibration reproduces the saved artefacts exactly. Interpretation for the reviewer:
the pooled parameter trends hold in **direction** across both formats (store area ↑, chain loyalty ↑,
distance sensitivity ↓), but **chain loyalty is a department-store phenomenon** (+34–36%) and barely
moves for general merchandise (+3–5%) — a genuine format-level difference, not a pooling artefact.

In [10]:
print('='*60); print('NOTEBOOK 14 VERIFICATION SUMMARY'); print('='*60)
nfail=sum(1 for _,ok,_ in PASS if not ok)
for name,ok,detail in PASS: print(('PASS ' if ok else 'FAIL ')+name)
print('-'*60); print(f'{len(PASS)-nfail}/{len(PASS)} checks passed')
assert nfail==0, f'{nfail} verification checks FAILED — investigate before trusting notebook'

NOTEBOOK 14 VERIFICATION SUMMARY
PASS mapping counts 88/154/40
PASS per-store match_type identical to saved
PASS per-store naics identical to saved
PASS eligible pools 4997 / 5276
PASS sample sizes 448 / 450
PASS 452210 sampled CBG set matches saved
PASS 452319 sampled CBG set matches saved
PASS 452210 PSO params reproduce saved CSVs (allclose)
PASS 452319 PSO params reproduce saved CSVs (allclose)
PASS 452210 fit matches saved report
PASS 452319 fit matches saved report
PASS 452210 2019->2020 %changes match report
PASS 452210 2019->2021 %changes match report
PASS 452319 2019->2020 %changes match report
PASS 452319 2019->2021 %changes match report
------------------------------------------------------------
15/15 checks passed
